# Step 5b — MCP (Model Context Protocol)

🇬🇧 **English** (this notebook)

MCP is an open standard (introduced by Anthropic in late 2024) for connecting AI agents to external tools and data sources through a common protocol, instead of every framework needing its own bespoke integration for every tool. Where step 5a's `SerperDevTool` is a tool built *into* `crewai_tools`, an MCP server is a **separate process** — potentially written in any language, by anyone — that exposes tools over a standard interface. CrewAI connects to it and uses whatever tools it offers.

## Background

> Anthropic (2024). *Introducing the Model Context Protocol*. [modelcontextprotocol.io](https://modelcontextprotocol.io)

Think of MCP like a USB-C port for AI applications: a standardized way to plug an agent into any compliant server exposing tools, data, or workflows — the agent doesn't need custom integration code per tool, and the server doesn't need to know anything about which AI framework is calling it.

## The exercise

This notebook spawns a small, official reference MCP server (`mcp-server-fetch` — fetches and reads web pages) as a local subprocess via `uvx`, and connects the agent to it. No extra installation needed — `uvx` downloads and runs the server on first use (it's part of this project's `uv` toolchain already).

Same Researcher role again — but instead of searching broadly like step 5a, it fetches and quotes one specific, canonical source directly: the European Commission's official AI Act page.

In [ ]:
import os

from dotenv import load_dotenv
from crewai import Agent
from crewai.mcp import MCPServerStdio

load_dotenv()

# ── The MCP server — a separate process, connected over stdio ────────────────
fetch_server = MCPServerStdio(command="uvx", args=["mcp-server-fetch"])

# ── Same "researcher" template as agents.yaml, {topic} filled in via an f-string ─
topic = "EU AI Act"

role      = f"{topic} Senior Data Researcher"
goal      = f"Uncover cutting-edge developments in {topic}"
backstory = (
    f"You're a seasoned researcher with a knack for uncovering the latest "
    f"developments in {topic}. Known for your ability to find the most relevant "
    f"information and present it in a clear and concise manner."
)

agent = Agent(
    role=role,
    goal=goal,
    backstory=backstory,
    mcps=[fetch_server],
    verbose=True,
)

# ── Ask it to fetch and use a specific URL ────────────────────────────────────
url = "https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai"
question = (
    f"Fetch {url} and list the exact obligations and dates it states for high-risk "
    "AI systems, quoting the page directly rather than paraphrasing from memory."
)

result = await agent.kickoff(question)
print(result.raw)

## Your task

1. Run the cell. The first run may take a few extra seconds while `uvx` downloads the server. Check the verbose log: can you see the MCP tool being called, and with what arguments?

2. Compare to step 5a: both let the agent reach outside its training data, but 5a searches broadly and summarizes across sources, while this one fetches and quotes exactly one page. Which answer do you trust more, and why?

3. Swap in your own team's topic — including a real URL relevant to it (e.g. the official regulation text, a Wikipedia page, a company's about page).

4. Fill in the **Step 5b** section of `EVALUATION.md`.